# 03 · Churn Modeling Baseline
Compare logistic regression and random forest on the cleaned historical data. This is a baseline exercise, not a production or causal model.

The target is `Attrition_Flag`; `CLIENTNUM` and any `Naive_Bayes_Classifier*` fields are excluded to avoid identifier memorization and target leakage. Stratified train, validation, and test splits are used. Model selection uses validation average precision because attrited customers are the minority class. The selected model is refit on the full training split and evaluated once on the untouched test split. Threshold metrics are shown at 0.50 for a first comparison; retention decisions need a separately chosen business threshold.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
project_root = Path('..').resolve()
sys.path.insert(0, str(project_root / 'scripts'))
from train_churn_model import run_training

In [ ]:
validation_results, summary = run_training()
display(validation_results)
print('Selected model:', summary['selected_model'])
print('Selected model test metrics:', summary['test_metrics_selected_model'])
print('Fit / validation / test rows:', summary['fit_rows_per_candidate'], '/', summary['validation_rows'], '/', summary['test_rows'])
print('Positive class:', summary['positive_class'])
print('Excluded features:', summary['excluded_features'])

## How to read the results
- **Average precision** summarizes the precision–recall tradeoff across thresholds and is the selection metric for this imbalanced target. Candidate selection uses the validation split; final metrics come from the untouched test split.
- **ROC AUC** measures ranking across thresholds; compare it with average precision rather than using it alone.
- Precision and recall use a 0.50 probability threshold. Lowering the threshold may catch more attrited customers and also flag more existing customers.
- Before deployment, consider time-aware validation if event dates become available, calibration, cost-based threshold selection, and operational review.

The selected fitted pipeline is written to `models/churn_pipeline.joblib`, and validation/test metrics and split details are written to `reports/model_evaluation.json`.